In [8]:
# CELL 1 — Environment Setup (Updated for Colab 2025)
import os

# Fix: update package list first, then install with fallback flag
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless --fix-missing -qq

# Upgrade to PySpark 4.0 to match Colab's environment
!pip install pyspark==4.0.0 --quiet

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Verify Java is actually there
java_check = os.popen("java -version 2>&1").read()
print(f"☕ Java: {java_check}")
print("✅ Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provid

In [9]:
# CELL 2 — SparkSession
from pyspark.sql import SparkSession
import os

# Updated for Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .appName("LocalPulseAI") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark version: {spark.version}")
print("✅ SparkSession is live — LocalPulse AI engine started.")

✅ Spark version: 4.0.0
✅ SparkSession is live — LocalPulse AI engine started.


In [4]:
# CELL 3 — Smoke Test
test_data = [('review_1', 'Great tacos, loved the place!'),
             ('review_2', 'Terrible service, never coming back.'),
             ('review_3', 'Average food, nothing special.')]
columns = ['review_id', 'review_text']
df_test = spark.createDataFrame(test_data, columns)
df_test.show(truncate=False)
print(f'✅ Smoke test passed — {df_test.count()} rows processed by Spark.')

+---------+------------------------------------+
|review_id|review_text                         |
+---------+------------------------------------+
|review_1 |Great tacos, loved the place!       |
|review_2 |Terrible service, never coming back.|
|review_3 |Average food, nothing special.      |
+---------+------------------------------------+

✅ Smoke test passed — 3 rows processed by Spark.


In [6]:
# CELL 4 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify your file is visible to Colab
import os

file_path = "/content/drive/MyDrive/LocalPulseAI/yelp_academic_dataset_review.json"

if os.path.exists(file_path):
    size = os.path.getsize(file_path) / (1024**3)
    print(f"✅ Dataset found — {size:.2f} GB")
else:
    print("❌ File not found — check your Drive path")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset found — 4.98 GB


In [10]:
# CELL 5 — Raw Data Ingestion

print("⏳ Loading Yelp reviews into Spark...")

df_raw = spark.read.json(file_path)

print(f"\n📊 Total reviews loaded: {df_raw.count():,}")
print(f"📋 Total columns: {len(df_raw.columns)}")
print(f"\n🔍 Schema:")
df_raw.printSchema()
print(f"\n👀 Sample rows:")
df_raw.show(5, truncate=50)

⏳ Loading Yelp reviews into Spark...

📊 Total reviews loaded: 6,990,280
📋 Total columns: 9

🔍 Schema:
root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)


👀 Sample rows:
+----------------------+----+-------------------+-----+----------------------+-----+--------------------------------------------------+------+----------------------+
|           business_id|cool|               date|funny|             review_id|stars|                                              text|useful|               user_id|
+----------------------+----+-------------------+-----+----------------------+-----+--------------------------------------------------+------+----------------------+
|XQfwVwDr-v0ZS3_CbbE5Xw|   0|2018-07-07 2